In [ ]:
import pandas as pd
import polars as pl

from superstore_sales_dataset.config import RAW_DATA_DIR

In [ ]:
sales_data = pl.read_csv(
    RAW_DATA_DIR / "superstore_sales_dataset.csv",
    encoding="utf8-lossy",
)

In [ ]:
sales_data.get_column("Order ID").unique()

# Auditoria Temporal

In [ ]:
sales_data.get_column("Order ID").unique()

In [ ]:
def polars_info(sales_data):
    print(f"Shape: {sales_data.shape}")
    print(f"Total de linhas: {sales_data.height}")
    print(f"Total de colunas: {sales_data.width}")
    print("\nInformações das colunas:\n")

    for col in sales_data.columns:
        print(
            f"{col}: "
            f"{sales_data[col].dtype}, "
            f"não nulos = {sales_data[col].count()}, "
            f"nulos = {sales_data[col].null_count()}"
        )


polars_info(sales_data)

In [ ]:
# Filtrando linhas com valores negativos na coluna 'Profit'
sales_data.filter(pl.any_horizontal(pl.col(["Profit"]) < 0)).head(3)

In [ ]:
# Contando o número de linhas com valores negativos na coluna 'Profit'
sales_data.select((pl.col("Profit") < 0).sum().alias("Profit")).head(3)

In [ ]:
# Criando uma nova coluna para indicar se a linha é de prejuízo (Profit < 0)
sales_data = sales_data.with_columns(
    pl.when(pl.col("Sales") != 0)
    .then(pl.col("Profit") / pl.col("Sales"))
    .otherwise(None)
    .alias("Profit Margin Ratio")
)

In [ ]:
# Criando uma nova coluna para indicar se a linha é de prejuízo (Profit < 0)
sales_data = sales_data.with_columns(
    pl.any_horizontal(pl.col(["Profit"]) < 0).alias("Is Loss Making")
)

## Primeira Visão do Sangramento (Pergunta Orientadora)

In [ ]:
# Calculando a porcentagem de pedidos que resultaram em prejuízo
loss_percentage = (
    sales_data.filter(pl.col("Is Loss Making")).height / sales_data.height
) * 100
print(f"Percentage of loss-making orders: {loss_percentage:.2f}%")

In [ ]:
# Percentual do faturamento total drenado por operações deficitárias
total_sales = sales_data.select(pl.col("Sales").sum()).to_numpy()[0][0]
total_loss_sales = (
    sales_data.filter(pl.col("Is Loss Making"))
    .select(pl.col("Sales").sum())
    .to_numpy()[0][0]
)
loss_sales_percentage = (total_loss_sales / total_sales) * 100

print(f"Total sales: {total_sales:,.2f}")
print(f"Total loss sales: {total_loss_sales:,.2f}")
print(
    f"Percentage of total sales from loss-making orders: {loss_sales_percentage:.2f}%"
)
print(
    f"Total orders: {sales_data.select(pl.col('Order ID').n_unique()).to_numpy()[0][0]}"
)